In [37]:
import scipy as sp
import numpy as np
import time as t

def pretty(d, indent=0): # in order to pretty print dictionaries
   for key, value in d.items():
      print('\t' * indent + str(key))
      if isinstance(value, dict):
         pretty(value, indent+1)
      else:
         print('\t' * (indent+1) + str(value))

def rel_err(x_sol, x):
    return f"The relative error is {np.linalg.norm(x-x_sol) / np.linalg.norm(x_sol)}"

def function(A, y, x_true, options=[]) :
    res = {}
    
    if "solve" in options :
        now = t.time()
        x = np.linalg.solve(A,y)
        res["solve"] = (rel_err(x_true, x), f"Time spent: {t.time()-now}s")
    
    if "splitting" in options :
        now = t.time()
        P, L, U = sp.linalg.lu(A, permute_l=False)
        z = sp.linalg.solve_triangular(L, P.T@y, lower=True)
        x = sp.linalg.solve_triangular(U, z, lower=False)
        res["splitting"] = (rel_err(x_true, x), f"Time spent: {t.time()-now}s")
    
    if "cholesky" in options :
        now = t.time()
        L = np.linalg.cholesky(A)
        z = sp.linalg.solve_triangular(L, y, lower=True)
        x = sp.linalg.solve_triangular(L.T, z, lower=False)
        res["cholesky"] = (rel_err(x_true, x), f"Time spent: {t.time()-now}s")
    
    return res 

In [38]:
n = 1000
A = np.random.rand(n,n)
x_true = np.ones(n)
y = A @ x_true

pretty(function(A, y, x_true, ["solve", "splitting"]))

solve
	('The relative error is 1.009181966838313e-12', 'Time spent: 0.0288238525390625s')
splitting
	('The relative error is 2.1370187035662472e-12', 'Time spent: 0.057888031005859375s')


In [39]:
n = 10
A = sp.linalg.hilbert(n)
x_true = np.ones(n)
y = A @ x_true

pretty(function(A, y, x_true, ["solve", "splitting", "cholesky"]))

solve
	('The relative error is 8.67039023709691e-05', 'Time spent: 0.00011587142944335938s')
splitting
	('The relative error is 7.279169219679924e-05', 'Time spent: 0.0005900859832763672s')
cholesky
	('The relative error is 0.00024552890914515864', 'Time spent: 0.0001480579376220703s')
